In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# --- Configuration ---
# Replace with your Phi-4 model path
model_name = "/data/projects/mayapeng/hosptials_model/Phi-4/pretrain/"
dataset_path = "/data/projects/mayapeng/hosptials_model/DeepSeek-R1-Distill-Qwen-14B/train.json"
output_dir = "/data/projects/mayapeng/hosptials_model/Phi-4/train_lora_res/"

# --- 1. Load model and tokenizer, and fix the tokenizer ---
print("🔄 Loading Phi-4 model...")

# Note: Phi-4's tokenizer has a known bug, need to manually fix pad_token and eos_token
# Explicitly set it immediately after loading the tokenizer, or download the fixed version in advance
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Fix Tokenizer
# Set the correct eos_token (should be <|im_end|>)
if tokenizer.eos_token != "<|im_end|>":
    tokenizer.eos_token = "<|im_end|>"
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

# Set a dedicated padding token to avoid duplication with eos_token
# Use an untrained dummy token
dummy_pad_token = "<|dummy_87|>"
tokenizer.add_tokens([dummy_pad_token])
tokenizer.pad_token = dummy_pad_token
tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(dummy_pad_token)

# Enable use_fast=True for better performance
tokenizer.use_fast = True

# Load model, bfloat16 is recommended to balance performance and precision
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,  # Phi-4 may rely on custom code
)

# Enable gradient checkpointing to save GPU memory
model.gradient_checkpointing_enable()
print("✅ Model loaded successfully.")

# --- 2. Data Preprocessing (Keep original data format, adjust chat template) ---
def format_to_messages(instruction, input_text, output_text):
    """Format data into a message list suitable for Phi-4"""
    if input_text:
        user_content = f"{instruction}\n{input_text}"
    else:
        user_content = instruction
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text}
    ]
    return messages

def preprocess_function(examples):
    instructions = examples["instruction"]
    inputs = examples.get("input", [""] * len(instructions))
    outputs = examples["output"]
    
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        messages = format_to_messages(inst, inp, out)
        # Phi-4's official chat template needs to be applied
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,   # No generation prompt needed during training
        )
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=5120,          # Adjust based on data length, note that Phi-4 supports 16k
        return_tensors=None,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# --- 3. Load dataset ---
print("🔄 Loading dataset...")
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"✅ Dataset loaded successfully, total {len(dataset)} records.")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("🔄 Tokenizing data...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)
print("✅ Data tokenization completed.")

# --- 4. LoRA target modules adapted to Phi-4 model structure ---
# Phi-4 is based on a Transformer decoder, its linear layer names are similar to Llama but need confirmation
# According to torchtune docs, Phi-4's attention module linear layers include q_proj, k_proj, v_proj, output_proj
# Meanwhile, MLP layers can also be fine-tuned, typically including gate_proj, up_proj, down_proj
target_modules = [
    "q_proj", "k_proj", "v_proj", "output_proj",  # Attention projection layers
    "gate_proj", "up_proj", "down_proj",          # MLP layers
]

lora_config = LoraConfig(
    r=32,                        # Set rank to 32 as specified in the criteria
    lora_alpha=64,               # Scaling factor set to 64 as specified in the criteria
    target_modules=target_modules,
    # Note: Phi-4 may not need to save embed_tokens and lm_head, but keeping them does no harm
    modules_to_save=["embed_tokens", "lm_head"],  # Fully fine-tune embedding layer and language modeling head
    lora_dropout=0.1,            # Dropout rate 0.1
    bias="all",                  # Train all bias terms as specified in the criteria
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ LoRA configuration completed.")

# --- 5. Training Arguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,                  # 3 epochs
    per_device_train_batch_size=2,       # Per-device batch size 2
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,       # Gradient accumulation over 4 steps
    warmup_steps=100,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    learning_rate=2e-4,                  # Learning rate 2×10⁻⁴
    fp16=False,                          # If GPU does not support BF16, can change to True
    bf16=True,                           # Recommended if GPU supports BF16
    group_by_length=True,
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch",                 # AdamW optimizer
    disable_tqdm=True,
)

# --- 6. Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training completed!")

# --- 7. Save model ---
print("💾 Saving model...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ LoRA adapters saved to: {output_dir}")